## 01 - Bronze Ingestion Pipeline
Ingests raw data from JDBC (SQL Server) and reference data into the bronze layer.

**Source tables:** `cards`, `transactions`, `users_data` (JDBC)  
**Reference data:** MCC codes lookup, Fraud labels  
**Target schema:** `jrvs_databricks.bronze`

In [0]:
# JDBC Configuration
jdbc_url = "jdbc:sqlserver://jrvs.database.windows.net:1433;database=finance;encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"
jdbc_user = "rocky@jrvs"
jdbc_password = "Ninja$007"

connection_properties = {
    "user": jdbc_user,
    "password": jdbc_password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

CATALOG = "jrvs_databricks"
BRONZE_SCHEMA = f"{CATALOG}.bronze"
VOLUME_PATH = f"/Volumes/{CATALOG}/bronze/finance_data"

In [0]:
%sql
-- Create schemas for medallion architecture
CREATE SCHEMA IF NOT EXISTS jrvs_databricks.bronze;
CREATE SCHEMA IF NOT EXISTS jrvs_databricks.silver;
CREATE SCHEMA IF NOT EXISTS jrvs_databricks.gold;

-- Create volume for JSON reference data
CREATE VOLUME IF NOT EXISTS jrvs_databricks.bronze.finance_data;

In [0]:
# Ingest cards table from JDBC
cards_df = (spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "cards")
    .option("user", jdbc_user)
    .option("password", jdbc_password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)

cards_df.write.mode("overwrite").saveAsTable(f"{BRONZE_SCHEMA}.cards")
print(f"Cards ingested: {spark.table(f'{BRONZE_SCHEMA}.cards').count()} rows")

Cards ingested: 6146 rows


In [0]:
# Ingest transactions table from JDBC
transactions_df = (spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "transactions")
    .option("user", jdbc_user)
    .option("password", jdbc_password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .option("fetchsize", "100000")
    .load()
)

transactions_df.write.mode("overwrite").saveAsTable(f"{BRONZE_SCHEMA}.transactions")
print(f"Transactions ingested: {spark.table(f'{BRONZE_SCHEMA}.transactions').count()} rows")

Transactions ingested: 13305915 rows


In [0]:
# Ingest users_data table from JDBC
users_df = (spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "users_data")
    .option("user", jdbc_user)
    .option("password", jdbc_password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)

users_df.write.mode("overwrite").saveAsTable(f"{BRONZE_SCHEMA}.users_data")
print(f"Users ingested: {spark.table(f'{BRONZE_SCHEMA}.users_data').count()} rows")

Users ingested: 2000 rows


In [0]:
# Ingest MCC codes from uploaded JSON file
# File format: single JSON object {"5812": "Eating Places and Restaurants", ...}
import json

with open(f"/Volumes/{CATALOG}/bronze/finance_data/mcc_codes.json", "r") as f:
    mcc_raw = json.load(f)

mcc_codes_data = [(code, desc) for code, desc in mcc_raw.items()]

from pyspark.sql.types import StructType, StructField, StringType
mcc_schema = StructType([StructField("mcc", StringType()), StructField("mcc_description", StringType())])
mcc_df = spark.createDataFrame(mcc_codes_data, schema=mcc_schema)

mcc_df.write.mode("overwrite").saveAsTable(f"{BRONZE_SCHEMA}.mcc_codes")
print(f"MCC codes loaded: {spark.table(f'{BRONZE_SCHEMA}.mcc_codes').count()} codes")
display(spark.table(f"{BRONZE_SCHEMA}.mcc_codes").limit(5))

MCC codes loaded: 109 codes


mcc,mcc_description
5812,Eating Places and Restaurants
5541,Service Stations
7996,"Amusement Parks, Carnivals, Circuses"
5411,"Grocery Stores, Supermarkets"
4784,Tolls and Bridge Fees


In [0]:
# Ingest fraud labels from uploaded JSON file
# File format: {"target": {"transaction_id": "Yes"/"No", ...}}
import json

fraud_labels_path = f"/Volumes/{CATALOG}/bronze/finance_data/train_fraud_labels.json"

with open(fraud_labels_path, "r") as f:
    fraud_raw = json.load(f)

# Extract the target dict: {transaction_id: "Yes"/"No"}
target_dict = fraud_raw["target"]
fraud_data = [(int(tid), 1 if label == "Yes" else 0) for tid, label in target_dict.items()]

from pyspark.sql.types import StructType, StructField, LongType, IntegerType
fraud_schema = StructType([
    StructField("transaction_id", LongType()),
    StructField("is_fraud", IntegerType())
])
fraud_labels_df = spark.createDataFrame(fraud_data, schema=fraud_schema)

fraud_labels_df.write.mode("overwrite").saveAsTable(f"{BRONZE_SCHEMA}.fraud_labels")

fraud_count = spark.table(f"{BRONZE_SCHEMA}.fraud_labels").filter("is_fraud = 1").count()
total_count = spark.table(f"{BRONZE_SCHEMA}.fraud_labels").count()
print(f"Fraud labels saved: {total_count:,} total, {fraud_count:,} fraudulent ({fraud_count/total_count*100:.2f}%)")
display(spark.table(f"{BRONZE_SCHEMA}.fraud_labels").limit(5))

Fraud labels saved: 8,914,963 total, 13,332 fraudulent (0.15%)


transaction_id,is_fraud
19618127,0
21826507,0
22352483,0
14480144,0
8421111,0


In [0]:
# Summary of bronze tables
print("=" * 60)
print("BRONZE INGESTION COMPLETE")
print("=" * 60)
for table in ["cards", "transactions", "users_data", "mcc_codes", "fraud_labels"]:
    count = spark.table(f"{BRONZE_SCHEMA}.{table}").count()
    print(f"  {BRONZE_SCHEMA}.{table}: {count:,} rows")
print("=" * 60)

BRONZE INGESTION COMPLETE
  jrvs_databricks.bronze.cards: 6,146 rows
  jrvs_databricks.bronze.transactions: 13,305,915 rows
  jrvs_databricks.bronze.users_data: 2,000 rows
  jrvs_databricks.bronze.mcc_codes: 109 rows
  jrvs_databricks.bronze.fraud_labels: 8,914,963 rows
